# Week 10 — dSprites-only OOD Steering Notebook

## Goal
Test whether a simple latent steering recipe learned on **in-domain dSprites** generalizes to a small **out-of-distribution** split.

## Experimental design
We use dSprites factors:
- `shape ∈ {0,1,2}`
- `scale ∈ {0,...,5}`
- `orientation ∈ {0,...,39}`
- `posX ∈ {0,...,31}`
- `posY ∈ {0,...,31}`

We train on a subset of **orientations 0..19** and evaluate on:
- **ID test**: unseen groups, but still orientations `0..19`
- **OOD test**: unseen groups with held-out orientations `20..39`

In [1]:
%pip install -q torch torchvision matplotlib pandas tqdm scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import math
import time
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm import tqdm
from IPython.display import display, Image

In [3]:
import warnings
warnings.filterwarnings("ignore", message=".*can only test a child process.*")
warnings.filterwarnings("ignore", category=FutureWarning)

In [4]:
cfg: Dict[str, Any] = {
    "seed": 42,
    "device": "cuda",

    "paths": {
        "dsprites_npz_path": "/kaggle/input/datasets/galievilyas/dsprits/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz",
        "run_root": "runs",
    },

    "data": {
        # OOD split = held-out orientations
        "train_orientations": list(range(0, 20)),
        "ood_orientations": list(range(20, 40)),

        # group = (shape, orientation, posX, posY), each selected group contributes all 6 scales
        "train_groups": 3500,
        "val_groups": 600,
        "id_test_groups": 800,
        "ood_test_groups": 800,
    },

    "loader": {
        "batch_size": 128,
        "num_workers": 0,
        "pin_memory": True,
    },

    "vae": {
        "latent_dim": 24,
        "base_channels": 32,
        "epochs": 30,
        "lr": 8e-4,
        "beta": 0.7,
        "beta_warmup_epochs": 8,
        "free_bits": 0.02,
        "weight_decay": 1e-5,
        "grad_clip": 1.0,
        "recon": "bce",  # "bce" or "mse"
    },

    "heads": {
        "epochs": 12,
        "lr": 2e-3,
        "batch_size": 512,
    },

    "steering": {
        "alpha_grid": [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0],
        "tau_img_values": [3.0, 5.0, 7.0],
        "tau_shape_conf": 0.20,
        "lambda_img": 0.06,
        "lambda_shape": 0.80,
        "strict_shape_consistency": True,
    },

    "preview": {
        "n_samples": 8,
    },

    "notes": {
        "experiment_name": "week10_dsprites_ood_only",
    },
}

cfg

{'seed': 42,
 'device': 'cuda',
 'paths': {'dsprites_npz_path': '/kaggle/input/datasets/galievilyas/dsprits/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz',
  'run_root': 'runs'},
 'data': {'train_orientations': [0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19],
  'ood_orientations': [20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   39],
  'train_groups': 3500,
  'val_groups': 600,
  'id_test_groups': 800,
  'ood_test_groups': 800},
 'loader': {'batch_size': 128, 'num_workers': 0, 'pin_memory': True},
 'vae': {'latent_dim': 24,
  'base_channels': 32,
  'epochs': 30,
  'lr': 0.0008,
  'beta': 0.7,
  'beta_warmup_epochs': 8,
  'free_bits': 0.02,
  'weight_decay': 1e-05,
  'grad_clip': 1.0,
  'recon': 'bce'},
 'heads': {'epochs': 12, 'lr': 0.002, 'batch_size': 512},
 'steering': {'alpha_grid': [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1

In [5]:
# helpers
def get_device(name: str) -> torch.device:
    if name == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    worker_seed = (torch.initial_seed() + worker_id) % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def save_json(path: str, obj: dict) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def make_run_dir(root: str, tag: str) -> str:
    stamp = time.strftime("%Y%m%d_%H%M%S")
    path = Path(root) / f"{stamp}_{tag}"
    (path / "plots").mkdir(parents=True, exist_ok=True)
    (path / "tables").mkdir(parents=True, exist_ok=True)
    (path / "checkpoints").mkdir(parents=True, exist_ok=True)
    return str(path)

def require_manual_path(value: str, label: str) -> str:
    value = str(value).strip()
    if value == "" or value.upper().startswith("TODO"):
        raise ValueError(f"Fill the TODO path for {label} before running the notebook.")
    path = Path(value).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"Path for {label} does not exist: {path}")
    return str(path)

def normalize_direction(v: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32)
    n = float(np.linalg.norm(v))
    return v / max(n, eps)

def encode_mu(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    mu, _ = model.enc(x)
    return mu

def decode_sigmoid(model: nn.Module, z: torch.Tensor) -> torch.Tensor:
    return torch.sigmoid(model.dec(z))

def load_checkpoint_state(model: nn.Module, checkpoint_path: str, device: torch.device) -> nn.Module:
    payload = torch.load(checkpoint_path, map_location=device)
    state = payload["model"] if isinstance(payload, dict) and "model" in payload else payload
    model.load_state_dict(state)
    return model

#  dSprites dataset
class DSpritesSubset(Dataset):
    def __init__(self, npz_path: str, indices: np.ndarray):
        data = np.load(npz_path, allow_pickle=True, mmap_mode="r")
        self.imgs = data["imgs"]
        self.classes = data["latents_classes"]
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = int(self.indices[i])
        x = self.imgs[idx].astype(np.float32)[None, ...]
        c = self.classes[idx].astype(np.int64)
        return torch.from_numpy(x), torch.from_numpy(c)

def build_group_splits(
        npz_path: str,
        train_orientations: List[int],
        ood_orientations: List[int],
        train_groups: int,
        val_groups: int,
        id_test_groups: int,
        ood_test_groups: int,
        seed: int = 42,
    ) -> Dict[str, np.ndarray]:

    data = np.load(npz_path, allow_pickle=True, mmap_mode="r")
    classes = np.asarray(data["latents_classes"], dtype=np.int64)

    # factor order: color, shape, scale, orientation, posX, posY
    group_factors = classes[:, [1, 3, 4, 5]]   # shape, orientation, posX, posY
    uniq_groups, inverse = np.unique(group_factors, axis=0, return_inverse=True)

    train_mask = np.isin(uniq_groups[:, 1], np.asarray(train_orientations, dtype=np.int64))
    ood_mask = np.isin(uniq_groups[:, 1], np.asarray(ood_orientations, dtype=np.int64))

    train_pool = np.where(train_mask)[0]
    ood_pool = np.where(ood_mask)[0]

    rng = np.random.default_rng(seed)
    rng.shuffle(train_pool)
    rng.shuffle(ood_pool)

    need_train = int(train_groups) + int(val_groups) + int(id_test_groups)
    if need_train > len(train_pool):
        raise ValueError(f"Requested {need_train} ID groups, but only {len(train_pool)} are available.")
    if int(ood_test_groups) > len(ood_pool):
        raise ValueError(f"Requested {ood_test_groups} OOD groups, but only {len(ood_pool)} are available.")

    g_train = train_pool[: int(train_groups)]
    g_val = train_pool[int(train_groups): int(train_groups) + int(val_groups)]
    g_id_test = train_pool[int(train_groups) + int(val_groups): need_train]
    g_ood_test = ood_pool[: int(ood_test_groups)]

    def group_ids_to_indices(group_ids: np.ndarray) -> np.ndarray:
        return np.where(np.isin(inverse, group_ids))[0].astype(np.int64)

    out = {
        "train_idx": group_ids_to_indices(g_train),
        "val_idx": group_ids_to_indices(g_val),
        "id_test_idx": group_ids_to_indices(g_id_test),
        "ood_test_idx": group_ids_to_indices(g_ood_test),
        "meta": {
            "n_unique_groups": int(len(uniq_groups)),
            "n_train_pool_groups": int(len(train_pool)),
            "n_ood_pool_groups": int(len(ood_pool)),
        },
    }
    return out

def summarize_split(indices: np.ndarray, name: str) -> pd.DataFrame:
    return pd.DataFrame([{
        "split": name,
        "n_samples": int(len(indices)),
        "n_groups_approx": int(len(indices) // 6),
        }
    ]
    )

In [6]:
# models
class ResidualBlock(nn.Module):
    def __init__(self, channels: int, groups: int = 8):
        super().__init__()
        g = max(1, min(groups, channels))
        while channels % g != 0 and g > 1:
            g -= 1
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.GroupNorm(g, channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.GroupNorm(g, channels),
        )
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(x + self.block(x))

class DownBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        g = max(1, min(8, out_ch))
        while out_ch % g != 0 and g > 1:
            g -= 1
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
            ResidualBlock(out_ch, groups=g),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        g = max(1, min(8, out_ch))
        while out_ch % g != 0 and g > 1:
            g -= 1
        self.block = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
            ResidualBlock(out_ch, groups=g),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

class DSpritesEncoder(nn.Module):
    def __init__(self, latent_dim: int, base_channels: int = 32):
        super().__init__()
        c1, c2, c3, c4 = base_channels, base_channels * 2, base_channels * 4, base_channels * 8
        self.stem = nn.Sequential(
            nn.Conv2d(1, c1, 3, padding=1, bias=False),
            nn.GroupNorm(8 if c1 % 8 == 0 else 4, c1),
            nn.SiLU(inplace=True),
            ResidualBlock(c1),
        )
        self.net = nn.Sequential(
            DownBlock(c1, c2),   # 64 -> 32
            DownBlock(c2, c3),   # 32 -> 16
            DownBlock(c3, c4),   # 16 -> 8
            DownBlock(c4, c4),   # 8 -> 4
        )
        self.fc_mu = nn.Linear(c4 * 4 * 4, latent_dim)
        self.fc_lv = nn.Linear(c4 * 4 * 4, latent_dim)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.net(self.stem(x)).flatten(1)
        return self.fc_mu(h), self.fc_lv(h)

class DSpritesDecoder(nn.Module):
    def __init__(self, latent_dim: int, base_channels: int = 32):
        super().__init__()
        c1, c2, c3, c4 = base_channels, base_channels * 2, base_channels * 4, base_channels * 8
        self.fc = nn.Linear(latent_dim, c4 * 4 * 4)
        self.net = nn.Sequential(
            ResidualBlock(c4),
            UpBlock(c4, c4),     # 4 -> 8
            UpBlock(c4, c3),     # 8 -> 16
            UpBlock(c3, c2),     # 16 -> 32
            UpBlock(c2, c1),     # 32 -> 64
            nn.Conv2d(c1, 1, 3, padding=1),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        c4 = self.fc.out_features // (4 * 4)
        h = self.fc(z).view(z.size(0), c4, 4, 4)
        return self.net(h)

class DSpritesVAE(nn.Module):
    def __init__(self, latent_dim: int, base_channels: int = 32):
        super().__init__()
        self.enc = DSpritesEncoder(latent_dim, base_channels=base_channels)
        self.dec = DSpritesDecoder(latent_dim, base_channels=base_channels)

    def reparam(self, mu: torch.Tensor, lv: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * lv)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor):
        mu, lv = self.enc(x)
        z = self.reparam(mu, lv)
        x_logits = self.dec(z)
        return x_logits, mu, lv

class ShapeHead(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.SiLU(inplace=True),
            nn.Dropout(0.10),
            nn.Linear(128, 64),
            nn.SiLU(inplace=True),
            nn.Linear(64, 3),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)

class ScaleHead(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.SiLU(inplace=True),
            nn.Dropout(0.10),
            nn.Linear(128, 64),
            nn.SiLU(inplace=True),
            nn.Linear(64, 6),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


def dsprites_vae_loss(
        x: torch.Tensor,
        x_logits: torch.Tensor,
        mu: torch.Tensor,
        lv: torch.Tensor,
        beta: float = 1.0,
        recon: str = "bce",
        free_bits: float = 0.0,
    ):
    if recon == "bce":
        rec = F.binary_cross_entropy_with_logits(x_logits, x, reduction="sum") / x.size(0)
    elif recon == "mse":
        rec = F.mse_loss(torch.sigmoid(x_logits), x, reduction="sum") / x.size(0)
    else:
        raise ValueError(f"unknown recon: {recon}")

    kl_per_dim = 0.5 * (mu.pow(2) + lv.exp() - 1.0 - lv)
    if free_bits > 0:
        kl = kl_per_dim.mean(dim=0).clamp_min(float(free_bits)).sum()
    else:
        kl = kl_per_dim.sum(dim=1).mean()

    return rec + beta * kl, rec, kl


@torch.no_grad()
def encode_dataset(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    zs, classes = [], []
    model.eval()
    for x, c in tqdm(loader, desc="encode split", leave=False):
        x = x.to(device, non_blocking=True)
        z = encode_mu(model, x)
        zs.append(z.cpu().numpy())
        classes.append(c.numpy())
    return np.concatenate(zs, axis=0), np.concatenate(classes, axis=0)


def build_latent_loader(z: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    ds = TensorDataset(torch.from_numpy(z).float(), torch.from_numpy(y).long())
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def train_dsprites_vae(
        train_loader: DataLoader,
        val_loader: DataLoader,
        run_dir: str,
        latent_dim: int,
        base_channels: int,
        epochs: int,
        lr: float,
        beta: float,
        beta_warmup_epochs: int,
        free_bits: float,
        weight_decay: float,
        grad_clip: float,
        recon: str,
        device: torch.device,
    ) -> Tuple[nn.Module, pd.DataFrame]:

    model = DSpritesVAE(latent_dim, base_channels=base_channels).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, int(epochs)), eta_min=lr * 0.1)

    rows = []
    best_val = float("inf")
    best_path = str(Path(run_dir) / "checkpoints" / "dsprites_vae_best.pt")

    for ep in range(1, int(epochs) + 1):
        beta_t = float(beta) * min(1.0, ep / max(1, int(beta_warmup_epochs)))

        model.train()
        tr_loss = tr_rec = tr_kl = 0.0
        n_tr = 0

        for x, _ in tqdm(train_loader, desc=f"VAE train ep{ep}", leave=False):
            x = x.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            x_logits, mu, lv = model(x)
            loss, rec, kl = dsprites_vae_loss(
                x, x_logits, mu, lv,
                beta=beta_t,
                recon=recon,
                free_bits=free_bits,
            )
            loss.backward()
            if grad_clip and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

            bs = x.size(0)
            n_tr += bs
            tr_loss += float(loss.item()) * bs
            tr_rec += float(rec.item()) * bs
            tr_kl += float(kl.item()) * bs

        model.eval()
        va_loss = va_rec = va_kl = 0.0
        n_va = 0
        with torch.no_grad():
            for x, _ in tqdm(val_loader, desc=f"VAE val ep{ep}", leave=False):
                x = x.to(device, non_blocking=True)
                x_logits, mu, lv = model(x)
                loss, rec, kl = dsprites_vae_loss(
                    x, x_logits, mu, lv,
                    beta=beta_t,
                    recon=recon,
                    free_bits=free_bits,
                )
                bs = x.size(0)
                n_va += bs
                va_loss += float(loss.item()) * bs
                va_rec += float(rec.item()) * bs
                va_kl += float(kl.item()) * bs

        current_lr = opt.param_groups[0]["lr"]
        row = {
            "epoch": ep,
            "beta_t": beta_t,
            "lr": current_lr,
            "train_loss": tr_loss / max(1, n_tr),
            "train_rec": tr_rec / max(1, n_tr),
            "train_kl": tr_kl / max(1, n_tr),
            "val_loss": va_loss / max(1, n_va),
            "val_rec": va_rec / max(1, n_va),
            "val_kl": va_kl / max(1, n_va),
        }
        rows.append(row)

        if row["val_loss"] < best_val:
            best_val = row["val_loss"]
            torch.save({"model": model.state_dict(), "epoch": ep}, best_path)

        sched.step()

    best_model = DSpritesVAE(latent_dim, base_channels=base_channels).to(device)
    best_model = load_checkpoint_state(best_model, best_path, device)
    return best_model, pd.DataFrame(rows)


def train_latent_head(
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        run_dir: str,
        name: str,
        epochs: int,
        lr: float,
        device: torch.device,
    ) -> Tuple[nn.Module, pd.DataFrame]:
    
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    rows = []
    best_acc = -1.0
    best_path = str(Path(run_dir) / "checkpoints" / f"{name}_best.pt")

    for ep in range(1, int(epochs) + 1):
        model.train()
        tr_loss = 0.0
        n_tr = 0
        for z, y in train_loader:
            z = z.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            logits = model(z)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()

            bs = z.size(0)
            n_tr += bs
            tr_loss += float(loss.item()) * bs

        model.eval()
        va_loss = 0.0
        correct = 0.0
        n_va = 0
        with torch.no_grad():
            for z, y in val_loader:
                z = z.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                logits = model(z)
                loss = F.cross_entropy(logits, y)
                bs = z.size(0)
                n_va += bs
                va_loss += float(loss.item()) * bs
                correct += float((logits.argmax(dim=1) == y).float().sum().item())

        acc = correct / max(1, n_va)
        rows.append({
            "epoch": ep,
            "train_loss": tr_loss / max(1, n_tr),
            "val_loss": va_loss / max(1, n_va),
            "val_acc": acc,
        })

        if acc > best_acc:
            best_acc = acc
            torch.save({"model": model.state_dict(), "epoch": ep}, best_path)

    if isinstance(model, ShapeHead):
        fresh = ShapeHead(model.net[0].in_features).to(device)
    elif isinstance(model, ScaleHead):
        fresh = ScaleHead(model.net[0].in_features).to(device)
    else:
        raise TypeError("Unsupported head type.")
    fresh = load_checkpoint_state(fresh, best_path, device)
    return fresh, pd.DataFrame(rows)


@torch.no_grad()
def evaluate_head(model: nn.Module, loader: DataLoader, device: torch.device, label_name: str, split_name: str) -> pd.DataFrame:
    correct = 0.0
    n = 0
    rows = []
    for z, y in loader:
        z = z.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(z)
        pred = logits.argmax(dim=1)
        correct += float((pred == y).float().sum().item())
        n += y.numel()
    rows.append({
        "split": split_name,
        "target": label_name,
        "acc": correct / max(1, n),
        "n": int(n),
    })
    return pd.DataFrame(rows)


In [7]:
# steering
def expected_scale_from_logits(logits: torch.Tensor) -> torch.Tensor:
    p = torch.softmax(logits, dim=1)
    vals = torch.arange(logits.size(1), device=logits.device, dtype=logits.dtype)
    return (p * vals.view(1, -1)).sum(dim=1)


def build_global_scale_direction(z_train: np.ndarray, classes_train: np.ndarray) -> np.ndarray:
    # group factors = shape, orientation, posX, posY
    groups = {}
    for z, c in zip(z_train, classes_train):
        key = (int(c[1]), int(c[3]), int(c[4]), int(c[5]))
        scale = int(c[2])
        groups.setdefault(key, {})[scale] = z

    deltas = []
    for g in groups.values():
        for s in range(5):
            if s in g and (s + 1) in g:
                deltas.append(g[s + 1] - g[s])

    if len(deltas) == 0:
        raise RuntimeError("No adjacent scale pairs found in the training split.")
    d = np.mean(np.stack(deltas, axis=0), axis=0)
    return normalize_direction(d)


@torch.no_grad()
def sweep_fixed_alpha(
        vae: nn.Module,
        shape_head: nn.Module,
        scale_head: nn.Module,
        loader: DataLoader,
        direction: np.ndarray,
        alpha_grid: List[float],
        tau_shape_conf: float,
        lambda_img: float,
        lambda_shape: float,
        strict_shape_consistency: bool,
        split_name: str,
        device: torch.device,
    ) -> pd.DataFrame:

    d = torch.from_numpy(direction).to(device)
    acc = {float(a): {
        "split": split_name,
        "alpha": float(a),
        "n": 0,
        "shape_acc_sum": 0.0,
        "shape_consistency_sum": 0.0,
        "scale_gain_sum": 0.0,
        "img_l2_sum": 0.0,
        "shape_conf_drop_sum": 0.0,
        "objective_sum": 0.0,
        "feasible_sum": 0.0,
    } for a in alpha_grid}

    for x, c in tqdm(loader, desc=f"fixed sweep [{split_name}]", leave=False):
        x = x.to(device)
        c = c.to(device)
        shape_true = c[:, 1]

        z = encode_mu(vae, x)
        x_rec = decode_sigmoid(vae, z)

        shape_logits_rec = shape_head(z)
        scale_logits_rec = scale_head(z)
        shape_pred_rec = shape_logits_rec.argmax(dim=1)
        shape_prob_rec = torch.softmax(shape_logits_rec, dim=1).gather(1, shape_pred_rec[:, None]).squeeze(1)
        exp_scale_rec = expected_scale_from_logits(scale_logits_rec)

        for alpha in alpha_grid:
            alpha = float(alpha)
            z_s = z + alpha * d.unsqueeze(0)
            x_s = decode_sigmoid(vae, z_s)

            shape_logits_s = shape_head(z_s)
            scale_logits_s = scale_head(z_s)

            shape_pred_s = shape_logits_s.argmax(dim=1)
            shape_prob_s = torch.softmax(shape_logits_s, dim=1).gather(1, shape_pred_rec[:, None]).squeeze(1)
            exp_scale_s = expected_scale_from_logits(scale_logits_s)

            gain = exp_scale_s - exp_scale_rec
            img_l2 = torch.norm((x_s - x_rec).flatten(1), dim=1)
            conf_drop = (shape_prob_rec - shape_prob_s).clamp_min(0.0)

            shape_ok = shape_pred_s.eq(shape_true).float()
            shape_cons = shape_pred_s.eq(shape_pred_rec).float()
            feasible = (conf_drop <= float(tau_shape_conf))
            if strict_shape_consistency:
                feasible = feasible & shape_pred_s.eq(shape_pred_rec)

            objective = gain - float(lambda_img) * img_l2 - float(lambda_shape) * conf_drop

            row = acc[alpha]
            row["n"] += x.size(0)
            row["shape_acc_sum"] += float(shape_ok.sum().item())
            row["shape_consistency_sum"] += float(shape_cons.sum().item())
            row["scale_gain_sum"] += float(gain.sum().item())
            row["img_l2_sum"] += float(img_l2.sum().item())
            row["shape_conf_drop_sum"] += float(conf_drop.sum().item())
            row["objective_sum"] += float(objective.sum().item())
            row["feasible_sum"] += float(feasible.float().sum().item())

    rows = []
    for alpha in alpha_grid:
        row = dict(acc[float(alpha)])
        n = max(1, int(row.pop("n")))
        rows.append({
            "split": row["split"],
            "alpha": row["alpha"],
            "shape_acc": row["shape_acc_sum"] / n,
            "shape_consistency": row["shape_consistency_sum"] / n,
            "scale_gain": row["scale_gain_sum"] / n,
            "img_l2_to_recon": row["img_l2_sum"] / n,
            "shape_conf_drop": row["shape_conf_drop_sum"] / n,
            "objective": row["objective_sum"] / n,
            "feasible_rate": row["feasible_sum"] / n,
            "n": n,
        })
    return pd.DataFrame(rows)


@torch.no_grad()
def evaluate_auto_alpha(
        vae: nn.Module,
        shape_head: nn.Module,
        scale_head: nn.Module,
        loader: DataLoader,
        direction: np.ndarray,
        alpha_grid: List[float],
        tau_img_values: List[float],
        tau_shape_conf: float,
        lambda_img: float,
        lambda_shape: float,
        strict_shape_consistency: bool,
        split_name: str,
        device: torch.device,
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    
    d = torch.from_numpy(direction).to(device)
    summary_rows = []
    sample_rows = []

    for tau_img in tau_img_values:
        summary_rows.append({
            "split": split_name,
            "tau_img": float(tau_img),
            "n": 0,
            "shape_acc_sum": 0.0,
            "shape_consistency_sum": 0.0,
            "scale_gain_sum": 0.0,
            "img_l2_sum": 0.0,
            "shape_conf_drop_sum": 0.0,
            "alpha_star_sum": 0.0,
            "nonzero_alpha_sum": 0.0,
            "objective_sum": 0.0,
        })

    tau_to_idx = {float(r["tau_img"]): i for i, r in enumerate(summary_rows)}
    global_idx = 0

    for x, c in tqdm(loader, desc=f"auto alpha [{split_name}]", leave=False):
        x = x.to(device)
        c = c.to(device)
        shape_true = c[:, 1]

        z = encode_mu(vae, x)
        x_rec = decode_sigmoid(vae, z)

        shape_logits_rec = shape_head(z)
        scale_logits_rec = scale_head(z)
        shape_pred_rec = shape_logits_rec.argmax(dim=1)
        shape_prob_rec = torch.softmax(shape_logits_rec, dim=1).gather(1, shape_pred_rec[:, None]).squeeze(1)
        exp_scale_rec = expected_scale_from_logits(scale_logits_rec)

        bs = x.size(0)

        for tau_img in tau_img_values:
            tau_img = float(tau_img)

            best_obj = torch.zeros(bs, device=device)
            best_alpha = torch.zeros(bs, device=device)
            best_shape_ok = shape_pred_rec.eq(shape_true).float()
            best_shape_cons = torch.ones(bs, device=device)
            best_gain = torch.zeros(bs, device=device)
            best_img_l2 = torch.zeros(bs, device=device)
            best_conf_drop = torch.zeros(bs, device=device)
            best_shape_pred = shape_pred_rec.clone()

            for alpha in alpha_grid:
                alpha = float(alpha)
                if alpha == 0.0:
                    continue

                z_s = z + alpha * d.unsqueeze(0)
                x_s = decode_sigmoid(vae, z_s)

                shape_logits_s = shape_head(z_s)
                scale_logits_s = scale_head(z_s)

                shape_pred_s = shape_logits_s.argmax(dim=1)
                shape_prob_s = torch.softmax(shape_logits_s, dim=1).gather(1, shape_pred_rec[:, None]).squeeze(1)
                exp_scale_s = expected_scale_from_logits(scale_logits_s)

                gain = exp_scale_s - exp_scale_rec
                img_l2 = torch.norm((x_s - x_rec).flatten(1), dim=1)
                conf_drop = (shape_prob_rec - shape_prob_s).clamp_min(0.0)

                shape_ok = shape_pred_s.eq(shape_true)
                shape_cons = shape_pred_s.eq(shape_pred_rec)

                feasible = (img_l2 <= tau_img) & (conf_drop <= float(tau_shape_conf))
                if strict_shape_consistency:
                    feasible = feasible & shape_cons

                obj = gain - float(lambda_img) * img_l2 - float(lambda_shape) * conf_drop
                update = feasible & (obj > best_obj)

                best_obj = torch.where(update, obj, best_obj)
                best_alpha = torch.where(update, torch.full_like(best_alpha, alpha), best_alpha)
                best_shape_ok = torch.where(update, shape_ok.float(), best_shape_ok)
                best_shape_cons = torch.where(update, shape_cons.float(), best_shape_cons)
                best_gain = torch.where(update, gain, best_gain)
                best_img_l2 = torch.where(update, img_l2, best_img_l2)
                best_conf_drop = torch.where(update, conf_drop, best_conf_drop)
                best_shape_pred = torch.where(update, shape_pred_s, best_shape_pred)

            row = summary_rows[tau_to_idx[tau_img]]
            row["n"] += bs
            row["shape_acc_sum"] += float(best_shape_ok.sum().item())
            row["shape_consistency_sum"] += float(best_shape_cons.sum().item())
            row["scale_gain_sum"] += float(best_gain.sum().item())
            row["img_l2_sum"] += float(best_img_l2.sum().item())
            row["shape_conf_drop_sum"] += float(best_conf_drop.sum().item())
            row["alpha_star_sum"] += float(best_alpha.sum().item())
            row["nonzero_alpha_sum"] += float((best_alpha != 0).float().sum().item())
            row["objective_sum"] += float(best_obj.sum().item())

            for i in range(bs):
                sample_rows.append({
                    "split": split_name,
                    "sample_idx": int(global_idx + i),
                    "tau_img": tau_img,
                    "alpha_star": float(best_alpha[i].item()),
                    "shape_true": int(shape_true[i].item()),
                    "shape_rec": int(shape_pred_rec[i].item()),
                    "shape_edit": int(best_shape_pred[i].item()),
                    "shape_acc": float(best_shape_ok[i].item()),
                    "shape_consistency": float(best_shape_cons[i].item()),
                    "scale_gain": float(best_gain[i].item()),
                    "img_l2_to_recon": float(best_img_l2[i].item()),
                    "shape_conf_drop": float(best_conf_drop[i].item()),
                    "objective": float(best_obj[i].item()),
                })

        global_idx += bs

    summary_df = pd.DataFrame([{
        "split": r["split"],
        "tau_img": r["tau_img"],
        "shape_acc": r["shape_acc_sum"] / max(1, r["n"]),
        "shape_consistency": r["shape_consistency_sum"] / max(1, r["n"]),
        "scale_gain": r["scale_gain_sum"] / max(1, r["n"]),
        "img_l2_to_recon": r["img_l2_sum"] / max(1, r["n"]),
        "shape_conf_drop": r["shape_conf_drop_sum"] / max(1, r["n"]),
        "alpha_star_mean": r["alpha_star_sum"] / max(1, r["n"]),
        "alpha_nonzero_rate": r["nonzero_alpha_sum"] / max(1, r["n"]),
        "objective": r["objective_sum"] / max(1, r["n"]),
        "n": int(r["n"]),
    } for r in summary_rows])

    sample_df = pd.DataFrame(sample_rows)
    return summary_df, sample_df


def pick_best_fixed_alpha(df_fixed: pd.DataFrame, split_name: str) -> float:
    x = df_fixed[df_fixed["split"] == split_name].copy()
    if len(x) == 0:
        raise ValueError(f"No fixed-alpha rows for split={split_name}")
    x = x.sort_values(["objective", "shape_consistency", "scale_gain"], ascending=[False, False, False])
    return float(x.iloc[0]["alpha"])


def pick_best_tau(df_auto: pd.DataFrame, split_name: str) -> float:
    x = df_auto[df_auto["split"] == split_name].copy()
    if len(x) == 0:
        raise ValueError(f"No auto-alpha rows for split={split_name}")
    x = x.sort_values(["objective", "shape_consistency", "scale_gain"], ascending=[False, False, False])
    return float(x.iloc[0]["tau_img"])

In [8]:
# plots
def plot_vae_curves(df: pd.DataFrame, out_png: str) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(10, 7))

    axes[0, 0].plot(df["epoch"], df["train_loss"], label="train_loss")
    axes[0, 0].plot(df["epoch"], df["val_loss"], label="val_loss")
    axes[0, 0].set_title("Total loss")
    axes[0, 0].set_xlabel("epoch")
    axes[0, 0].grid(alpha=0.25)
    axes[0, 0].legend()

    axes[0, 1].plot(df["epoch"], df["train_rec"], label="train_rec")
    axes[0, 1].plot(df["epoch"], df["val_rec"], label="val_rec")
    axes[0, 1].plot(df["epoch"], df["train_kl"], label="train_kl")
    axes[0, 1].plot(df["epoch"], df["val_kl"], label="val_kl")
    axes[0, 1].set_title("Recon / KL")
    axes[0, 1].set_xlabel("epoch")
    axes[0, 1].grid(alpha=0.25)
    axes[0, 1].legend()

    if "beta_t" in df.columns:
        axes[1, 0].plot(df["epoch"], df["beta_t"], label="beta_t")
        axes[1, 0].set_title("KL warmup")
        axes[1, 0].set_xlabel("epoch")
        axes[1, 0].grid(alpha=0.25)

    if "lr" in df.columns:
        axes[1, 1].plot(df["epoch"], df["lr"], label="lr")
        axes[1, 1].set_title("Learning rate")
        axes[1, 1].set_xlabel("epoch")
        axes[1, 1].grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def plot_head_curves(df_shape: pd.DataFrame, df_scale: pd.DataFrame, out_png: str) -> None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(df_shape["epoch"], df_shape["val_acc"], label="shape val acc")
    ax.plot(df_scale["epoch"], df_scale["val_acc"], label="scale val acc")
    ax.set_xlabel("epoch")
    ax.set_ylabel("accuracy")
    ax.set_title("Latent head validation accuracy")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


def plot_fixed_tradeoff(df_fixed: pd.DataFrame, out_png: str) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for split_name in sorted(df_fixed["split"].unique()):
        x = df_fixed[df_fixed["split"] == split_name].sort_values("alpha")
        ax.plot(x["img_l2_to_recon"], x["scale_gain"], marker="o", label=split_name)
        for _, r in x.iterrows():
            ax.text(r["img_l2_to_recon"], r["scale_gain"], f"{r['alpha']:.1f}", fontsize=8)
    ax.set_xlabel("image drift to reconstruction")
    ax.set_ylabel("expected scale gain")
    ax.set_title("Fixed-alpha tradeoff")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


def plot_auto_summary(df_auto: pd.DataFrame, out_png: str) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    markers = {"id_test": "o", "ood_test": "s"}
    for split_name in sorted(df_auto["split"].unique()):
        x = df_auto[df_auto["split"] == split_name].sort_values("tau_img")
        ax.plot(x["img_l2_to_recon"], x["scale_gain"], marker=markers.get(split_name, "o"), label=split_name)
        for _, r in x.iterrows():
            ax.text(r["img_l2_to_recon"], r["scale_gain"], f"τ={r['tau_img']:.0f}", fontsize=8)
    ax.set_xlabel("image drift to reconstruction")
    ax.set_ylabel("expected scale gain")
    ax.set_title("Auto-alpha tradeoff")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


def plot_auto_alpha_hist(df_samples: pd.DataFrame, tau_img: float, out_png: str) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    bins = np.asarray(sorted(df_samples["alpha_star"].unique()), dtype=np.float32)
    if len(bins) <= 1:
        bins = np.array([-0.25, 0.25])
    else:
        step = float(np.min(np.diff(np.unique(bins))))
        bins = np.concatenate([[bins.min() - step / 2], bins + step / 2])

    for split_name in sorted(df_samples["split"].unique()):
        x = df_samples[(df_samples["split"] == split_name) & (df_samples["tau_img"] == tau_img)]["alpha_star"].values
        ax.hist(x, bins=bins, alpha=0.55, label=split_name)
    ax.set_xlabel("alpha*")
    ax.set_ylabel("count")
    ax.set_title(f"Auto-alpha histogram (tau_img={tau_img})")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


@torch.no_grad()
def render_preview_grid(
        dataset: Dataset,
        out_png: str,
        vae: nn.Module,
        direction: np.ndarray,
        selected_rows: pd.DataFrame,
        device: torch.device,
    ) -> None:

    selected_rows = selected_rows.head(8).copy()
    if len(selected_rows) == 0:
        return

    d = torch.from_numpy(direction).to(device)
    fig, axes = plt.subplots(len(selected_rows), 3, figsize=(8, 2.2 * len(selected_rows)))
    if len(selected_rows) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, (_, r) in enumerate(selected_rows.iterrows()):
        x, _ = dataset[int(r["sample_idx"])]
        x = x.unsqueeze(0).to(device)
        z = encode_mu(vae, x)
        x_rec = decode_sigmoid(vae, z)
        z_s = z + float(r["alpha_star"]) * d.unsqueeze(0)
        x_s = decode_sigmoid(vae, z_s)

        imgs = [x.squeeze().cpu().numpy(), x_rec.squeeze().cpu().numpy(), x_s.squeeze().cpu().numpy()]
        titles = ["original", "recon", "steered"]
        for j in range(3):
            ax = axes[i, j]
            ax.imshow(imgs[j], cmap="gray")
            ax.axis("off")
            if i == 0:
                ax.set_title(titles[j], fontsize=11)

        axes[i, 0].set_ylabel(
            f"a*={r['alpha_star']:.2f}\n"
            f"gain={r['scale_gain']:.2f}\n"
            f"shape {int(r['shape_rec'])}->{int(r['shape_edit'])}",
            fontsize=9
        )

    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


def write_memo(
        out_path: str,
        split_df: pd.DataFrame,
        probe_df: pd.DataFrame,
        fixed_df: pd.DataFrame,
        auto_df: pd.DataFrame,
    ) -> None:
    
    best_fixed_id = fixed_df[fixed_df["split"] == "id_test"].sort_values("objective", ascending=False).iloc[0]
    best_fixed_ood = fixed_df[(fixed_df["split"] == "ood_test") & (fixed_df["alpha"] == best_fixed_id["alpha"])].iloc[0]

    best_auto_id = auto_df[auto_df["split"] == "id_test"].sort_values("objective", ascending=False).iloc[0]
    best_auto_ood = auto_df[(auto_df["split"] == "ood_test") & (auto_df["tau_img"] == best_auto_id["tau_img"])].iloc[0]

    shape_id = probe_df[(probe_df["split"] == "id_test") & (probe_df["target"] == "shape")]["acc"].iloc[0]
    shape_ood = probe_df[(probe_df["split"] == "ood_test") & (probe_df["target"] == "shape")]["acc"].iloc[0]
    scale_id = probe_df[(probe_df["split"] == "id_test") & (probe_df["target"] == "scale")]["acc"].iloc[0]
    scale_ood = probe_df[(probe_df["split"] == "ood_test") & (probe_df["target"] == "scale")]["acc"].iloc[0]

    lines = [
        "# Weekly memo — dSprites OOD extension",
        "",
        "## What was done",
        "- Trained a compact dSprites VAE on ID orientations only.",
        "- Trained latent heads for shape and scale.",
        "- Built one global latent direction for increasing scale.",
        "- Evaluated fixed-alpha and auto-alpha steering on ID and OOD splits.",
        "",
        "## Dataset split",
        split_df.to_markdown(index=False),
        "",
        "## Probe quality",
        probe_df.to_markdown(index=False),
        "",
        "## Main observations",
        f"- Shape probe accuracy: ID = {shape_id:.4f}, OOD = {shape_ood:.4f}.",
        f"- Scale probe accuracy: ID = {scale_id:.4f}, OOD = {scale_ood:.4f}.",
        f"- Best fixed alpha on ID: alpha = {best_fixed_id['alpha']:.2f}, scale gain = {best_fixed_id['scale_gain']:.4f}, shape consistency = {best_fixed_id['shape_consistency']:.4f}, drift = {best_fixed_id['img_l2_to_recon']:.4f}.",
        f"- Same fixed alpha on OOD: scale gain = {best_fixed_ood['scale_gain']:.4f}, shape consistency = {best_fixed_ood['shape_consistency']:.4f}, drift = {best_fixed_ood['img_l2_to_recon']:.4f}.",
        f"- Best auto setting on ID: tau_img = {best_auto_id['tau_img']:.2f}, scale gain = {best_auto_id['scale_gain']:.4f}, shape consistency = {best_auto_id['shape_consistency']:.4f}, alpha_nonzero_rate = {best_auto_id['alpha_nonzero_rate']:.4f}.",
        f"- Same auto setting on OOD: scale gain = {best_auto_ood['scale_gain']:.4f}, shape consistency = {best_auto_ood['shape_consistency']:.4f}, alpha_nonzero_rate = {best_auto_ood['alpha_nonzero_rate']:.4f}.",
        "",
        "## Interpretation",
        "- If OOD shape consistency stays close to ID while scale gain remains positive, the steering direction transfers beyond the training support.",
        "- If OOD gain collapses or shape consistency drops, failure is likely tied to representation shift across held-out orientations.",
        "",
        "## Risks",
        "- The result may depend strongly on VAE quality and latent disentanglement.",
        "- OOD here is small: only orientation support shifts, not a new dataset.",
        "- Fixed-alpha may over-edit on OOD even if it looks fine on ID.",
        "",
        "## Next step",
        "- Compare the same steering recipe against a stronger local direction or nearest-neighbor-conditioned direction on dSprites.",
    ]

    Path(out_path).write_text("\n".join(lines), encoding="utf-8")

In [9]:
# ------------------------ main ------------------------
set_seed(int(cfg["seed"]))
device = get_device(cfg["device"])

dsprites_npz_path = require_manual_path(cfg["paths"]["dsprites_npz_path"], "paths.dsprites_npz_path")
run_dir = make_run_dir(cfg["paths"]["run_root"], cfg["notes"]["experiment_name"])
save_json(str(Path(run_dir) / "config_used.json"), cfg)

print("device:", device)
print("run_dir:", run_dir)
print("dsprites:", dsprites_npz_path)

splits = build_group_splits(
    npz_path=dsprites_npz_path,
    train_orientations=list(cfg["data"]["train_orientations"]),
    ood_orientations=list(cfg["data"]["ood_orientations"]),
    train_groups=int(cfg["data"]["train_groups"]),
    val_groups=int(cfg["data"]["val_groups"]),
    id_test_groups=int(cfg["data"]["id_test_groups"]),
    ood_test_groups=int(cfg["data"]["ood_test_groups"]),
    seed=int(cfg["seed"]),
)

split_df = pd.concat([
    summarize_split(splits["train_idx"], "train"),
    summarize_split(splits["val_idx"], "val"),
    summarize_split(splits["id_test_idx"], "id_test"),
    summarize_split(splits["ood_test_idx"], "ood_test"),
], ignore_index=True)
display(split_df)

train_ds = DSpritesSubset(dsprites_npz_path, splits["train_idx"])
val_ds = DSpritesSubset(dsprites_npz_path, splits["val_idx"])
id_test_ds = DSpritesSubset(dsprites_npz_path, splits["id_test_idx"])
ood_test_ds = DSpritesSubset(dsprites_npz_path, splits["ood_test_idx"])

loader_cfg = cfg["loader"]
common_loader_kwargs = dict(
    batch_size=int(loader_cfg["batch_size"]),
    num_workers=int(loader_cfg["num_workers"]),
    pin_memory=bool(loader_cfg.get("pin_memory", False)) and device.type == "cuda",
    worker_init_fn=None if int(loader_cfg["num_workers"]) == 0 else seed_worker,
)

train_loader = DataLoader(train_ds, shuffle=True, **common_loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **common_loader_kwargs)
id_test_loader = DataLoader(id_test_ds, shuffle=False, **common_loader_kwargs)
ood_test_loader = DataLoader(ood_test_ds, shuffle=False, **common_loader_kwargs)

# train VAE
vae, vae_curve = train_dsprites_vae(
    train_loader=train_loader,
    val_loader=val_loader,
    run_dir=run_dir,
    latent_dim=int(cfg["vae"]["latent_dim"]),
    base_channels=int(cfg["vae"]["base_channels"]),
    epochs=int(cfg["vae"]["epochs"]),
    lr=float(cfg["vae"]["lr"]),
    beta=float(cfg["vae"]["beta"]),
    beta_warmup_epochs=int(cfg["vae"]["beta_warmup_epochs"]),
    free_bits=float(cfg["vae"]["free_bits"]),
    weight_decay=float(cfg["vae"]["weight_decay"]),
    grad_clip=float(cfg["vae"]["grad_clip"]),
    recon=str(cfg["vae"]["recon"]),
    device=device,
)
vae_curve.to_csv(Path(run_dir) / "tables" / "vae_curve.csv", index=False)

# encoding splits
eval_batch_size = int(loader_cfg["batch_size"])
eval_loader_kwargs = dict(
    batch_size=eval_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=bool(loader_cfg.get("pin_memory", False)) and device.type == "cuda",
)
train_eval_loader = DataLoader(train_ds, **eval_loader_kwargs)
val_eval_loader = DataLoader(val_ds, **eval_loader_kwargs)
id_eval_loader = DataLoader(id_test_ds, **eval_loader_kwargs)
ood_eval_loader = DataLoader(ood_test_ds, **eval_loader_kwargs)

z_train, c_train = encode_dataset(vae, train_eval_loader, device)
z_val, c_val = encode_dataset(vae, val_eval_loader, device)
z_id, c_id = encode_dataset(vae, id_eval_loader, device)
z_ood, c_ood = encode_dataset(vae, ood_eval_loader, device)

# factor order: color, shape, scale, orientation, posX, posY
y_shape_train, y_shape_val, y_shape_id, y_shape_ood = c_train[:, 1], c_val[:, 1], c_id[:, 1], c_ood[:, 1]
y_scale_train, y_scale_val, y_scale_id, y_scale_ood = c_train[:, 2], c_val[:, 2], c_id[:, 2], c_ood[:, 2]

# train latent heads
head_cfg = cfg["heads"]
shape_head, shape_curve = train_latent_head(
    ShapeHead(z_train.shape[1]),
    build_latent_loader(z_train, y_shape_train, batch_size=int(head_cfg["batch_size"]), shuffle=True),
    build_latent_loader(z_val, y_shape_val, batch_size=int(head_cfg["batch_size"]), shuffle=False),
    run_dir=run_dir,
    name="shape_head",
    epochs=int(head_cfg["epochs"]),
    lr=float(head_cfg["lr"]),
    device=device,
)

scale_head, scale_curve = train_latent_head(
    ScaleHead(z_train.shape[1]),
    build_latent_loader(z_train, y_scale_train, batch_size=int(head_cfg["batch_size"]), shuffle=True),
    build_latent_loader(z_val, y_scale_val, batch_size=int(head_cfg["batch_size"]), shuffle=False),
    run_dir=run_dir,
    name="scale_head",
    epochs=int(head_cfg["epochs"]),
    lr=float(head_cfg["lr"]),
    device=device,
)

shape_curve.to_csv(Path(run_dir) / "tables" / "shape_head_curve.csv", index=False)
scale_curve.to_csv(Path(run_dir) / "tables" / "scale_head_curve.csv", index=False)

probe_df = pd.concat([
    evaluate_head(shape_head, build_latent_loader(z_id, y_shape_id, batch_size=1024, shuffle=False), device, "shape", "id_test"),
    evaluate_head(shape_head, build_latent_loader(z_ood, y_shape_ood, batch_size=1024, shuffle=False), device, "shape", "ood_test"),
    evaluate_head(scale_head, build_latent_loader(z_id, y_scale_id, batch_size=1024, shuffle=False), device, "scale", "id_test"),
    evaluate_head(scale_head, build_latent_loader(z_ood, y_scale_ood, batch_size=1024, shuffle=False), device, "scale", "ood_test"),
], ignore_index=True)
probe_df.to_csv(Path(run_dir) / "tables" / "probe_eval.csv", index=False)
display(probe_df)

# build global scale direction
direction = build_global_scale_direction(z_train, c_train)
np.save(Path(run_dir) / "tables" / "global_scale_direction.npy", direction)

# Fixed-alpha sweep
steer_cfg = cfg["steering"]
df_fixed_id = sweep_fixed_alpha(
    vae, shape_head, scale_head, id_test_loader, direction,
    alpha_grid=list(steer_cfg["alpha_grid"]),
    tau_shape_conf=float(steer_cfg["tau_shape_conf"]),
    lambda_img=float(steer_cfg["lambda_img"]),
    lambda_shape=float(steer_cfg["lambda_shape"]),
    strict_shape_consistency=bool(steer_cfg["strict_shape_consistency"]),
    split_name="id_test",
    device=device,
)
df_fixed_ood = sweep_fixed_alpha(
    vae, shape_head, scale_head, ood_test_loader, direction,
    alpha_grid=list(steer_cfg["alpha_grid"]),
    tau_shape_conf=float(steer_cfg["tau_shape_conf"]),
    lambda_img=float(steer_cfg["lambda_img"]),
    lambda_shape=float(steer_cfg["lambda_shape"]),
    strict_shape_consistency=bool(steer_cfg["strict_shape_consistency"]),
    split_name="ood_test",
    device=device,
)
df_fixed = pd.concat([df_fixed_id, df_fixed_ood], ignore_index=True)
df_fixed.to_csv(Path(run_dir) / "tables" / "fixed_alpha_sweep.csv", index=False)

# auto-alpha
df_auto_id, df_samples_id = evaluate_auto_alpha(
    vae, shape_head, scale_head, id_test_loader, direction,
    alpha_grid=list(steer_cfg["alpha_grid"]),
    tau_img_values=list(steer_cfg["tau_img_values"]),
    tau_shape_conf=float(steer_cfg["tau_shape_conf"]),
    lambda_img=float(steer_cfg["lambda_img"]),
    lambda_shape=float(steer_cfg["lambda_shape"]),
    strict_shape_consistency=bool(steer_cfg["strict_shape_consistency"]),
    split_name="id_test",
    device=device,
)
df_auto_ood, df_samples_ood = evaluate_auto_alpha(
    vae, shape_head, scale_head, ood_test_loader, direction,
    alpha_grid=list(steer_cfg["alpha_grid"]),
    tau_img_values=list(steer_cfg["tau_img_values"]),
    tau_shape_conf=float(steer_cfg["tau_shape_conf"]),
    lambda_img=float(steer_cfg["lambda_img"]),
    lambda_shape=float(steer_cfg["lambda_shape"]),
    strict_shape_consistency=bool(steer_cfg["strict_shape_consistency"]),
    split_name="ood_test",
    device=device,
)
df_auto = pd.concat([df_auto_id, df_auto_ood], ignore_index=True)
df_samples = pd.concat([df_samples_id, df_samples_ood], ignore_index=True)
df_auto.to_csv(Path(run_dir) / "tables" / "auto_alpha_summary.csv", index=False)
df_samples.to_csv(Path(run_dir) / "tables" / "auto_alpha_samples.csv", index=False)

# best-setting comparison
best_fixed_alpha = pick_best_fixed_alpha(df_fixed, "id_test")
best_tau = pick_best_tau(df_auto, "id_test")

comparison_df = pd.concat([
    df_fixed[(df_fixed["split"].isin(["id_test", "ood_test"])) & (df_fixed["alpha"] == best_fixed_alpha)].assign(method=f"fixed_alpha_{best_fixed_alpha:.2f}"),
    df_auto[(df_auto["split"].isin(["id_test", "ood_test"])) & (df_auto["tau_img"] == best_tau)].assign(method=f"auto_tau_{best_tau:.2f}"),
], ignore_index=True)
comparison_df.to_csv(Path(run_dir) / "tables" / "selected_comparison.csv", index=False)

# plots
plot_dir = Path(run_dir) / "plots"
plot_vae_curves(vae_curve, str(plot_dir / "vae_curves.png"))
plot_head_curves(shape_curve, scale_curve, str(plot_dir / "head_curves.png"))
plot_fixed_tradeoff(df_fixed, str(plot_dir / "fixed_tradeoff.png"))
plot_auto_summary(df_auto, str(plot_dir / "auto_tradeoff.png"))
plot_auto_alpha_hist(df_samples, tau_img=best_tau, out_png=str(plot_dir / "auto_alpha_hist.png"))

preview_id = (
    df_samples[(df_samples["split"] == "id_test") & (df_samples["tau_img"] == best_tau)]
    .sort_values(["objective", "scale_gain"], ascending=[False, False])
    .head(int(cfg["preview"]["n_samples"]))
)
preview_ood = (
    df_samples[(df_samples["split"] == "ood_test") & (df_samples["tau_img"] == best_tau)]
    .sort_values(["objective", "scale_gain"], ascending=[False, False])
    .head(int(cfg["preview"]["n_samples"]))
)
render_preview_grid(id_test_ds, str(plot_dir / "preview_id.png"), vae, direction, preview_id, device)
render_preview_grid(ood_test_ds, str(plot_dir / "preview_ood.png"), vae, direction, preview_ood, device)

# 9) Memo
write_memo(
    out_path=str(Path(run_dir) / "memo.md"),
    split_df=split_df,
    probe_df=probe_df,
    fixed_df=df_fixed,
    auto_df=df_auto,
)

result = {
    "run_dir": run_dir,
    "best_fixed_alpha": best_fixed_alpha,
    "best_auto_tau_img": best_tau,
}

print("\nDone.")
print(json.dumps(result, indent=2))
display(comparison_df)

device: cpu
run_dir: runs/20260331_062508_week10_dsprites_ood_only
dsprites: /kaggle/input/datasets/galievilyas/dsprits/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz


,split,n_samples,n_groups_approx
0,train,21000,3500
1,val,3600,600
2,id_test,4800,800
3,ood_test,4800,800


VAE train ep1:   0%|          | 0/165 [00:00<?, ?it/s]

VAE train ep1:   1%|          | 1/165 [00:11<31:22, 11.48s/it]

VAE train ep1:   1%|          | 2/165 [00:21<29:18, 10.79s/it]

...

VAE val ep27:  97%|█████████▋| 28/29 [01:28<00:03,  3.17s/it]

VAE val ep27: 100%|██████████| 29/29 [01:28<00:00,  2.31s/it]

VAE train ep28:   0%|          | 0/165 [00:00<?, ?it/s]

VAE train ep28:   1%|          | 1/165 [00:10<28:30, 10.43s/it]

VAE train ep28:   1%|          | 2/165 [00:20<27:42, 10.20s/it]

In [ ]:
# quick preview of saved plots
plot_dir = Path(result["run_dir"]) / "plots"
for name in [
    "vae_curves.png",
    "head_curves.png",
    "fixed_tradeoff.png",
    "auto_tradeoff.png",
    "auto_alpha_hist.png",
    "preview_id.png",
    "preview_ood.png",
]:
    p = plot_dir / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))

In [ ]:
!zip -r output.zip /kaggle/working